# Request Input & Output Processing

Welcome to the first chapter in our journey to understand vLLM! Before a Large Language Model (LLM) can work its magic, we need to speak its language. By the end of this notebook, you will understand how a user's text prompt is translated into a format the model can process, and how the model's numerical output is translated back into human-readable text, especially for real-time streaming.

This is the fundamental "front door" and "back door" of any LLM serving system.

In [ ]:
# === Setup Cell ===
# All imports needed for this notebook.

from dataclasses import dataclass, field
from typing import List, Dict, Tuple
import time
from IPython.display import clear_output

# A simple mock tokenizer to simulate what a real tokenizer (like one from HuggingFace) does.
class MockTokenizer:
    """A mock tokenizer to convert between strings and token IDs."""
    def __init__(self):
        # A small, fixed vocabulary. In a real scenario, this would have 32,000+ entries.
        self.vocab = {
            "<|endoftext|>": 0, "Hello": 1, ",": 2, " world": 3, "!": 4, "This": 5,
            " is": 6, " a": 7, " test": 8, ".": 9, "The": 10, " quick": 11,
            " brown": 12, " fox": 13, " jumps": 14, " over": 15, " the": 16,
            " lazy": 17, " dog": 18
        }
        self.inverse_vocab = {v: k for k, v in self.vocab.items()}

    def encode(self, text: str) -> List[int]:
        """Converts a string of text into a list of token IDs."""
        # This is a very basic implementation. Real tokenizers are much more complex.
        # We'll split by space and handle punctuation separately for this example.
        import re
        words = re.findall(r'\w+|\S', text)
        return [self.vocab.get(word, self.vocab.get(f" {word}", -1)) for word in words]

    def decode(self, token_ids: List[int]) -> str:
        """Converts a list of token IDs back into a string."""
        # Simple lookup, then join. The lstrip() handles leading spaces on words.
        tokens = [self.inverse_vocab.get(id, "<UNK>") for id in token_ids]
        return "".join(tokens).lstrip()

@dataclass
class GenerationRequest:
    """A simplified representation of a user's request."""
    request_id: str
    prompt: str

@dataclass
class ProcessedRequest:
    """A request after it has been tokenized by the InputProcessor."""
    request_id: str
    prompt_token_ids: List[int]

@dataclass
class RequestOutput:
    """The final output for a request, after generation and detokenization."""
    request_id: str
    generated_text: str
    finished: bool = False

### The Core Idea: The Restaurant Analogy

Imagine an LLM serving engine as a high-end restaurant. A customer (the user) places an order in plain English: "I'd like the house special, please."

The waiter doesn't just shout this sentence into the kitchen. That would be chaotic and inefficient. Instead, a **sous-chef** acts as our `InputProcessor`.

1.  **Translate the Order (Tokenization):** The sous-chef takes the verbal order and translates it onto a standardized kitchen ticket. "House special" becomes a list of precise ingredients and steps: `[item_id: 7, prep_steps: [chop, saute, plate], modifiers: none]`. This is exactly what **tokenization** does: it converts the fuzzy, human-friendly prompt string into a list of precise, machine-readable integer IDs.

2.  **Execute the Recipe (Model Inference):** The head chef (our `EngineCore` and `Executor`) takes this ticket and executes the steps, cooking the dish. This is the model's forward pass, which generates a sequence of new ingredient IDs.

3.  **Plate and Serve (Detokenization):** The dish is cooked, but it's still in pots and pans. Another kitchen staffer, our `OutputProcessor`, takes the raw components and artfully plates them. It might add a sprig of parsley for garnish and ensure it looks appealing. For a streaming response, this is like the waiter bringing out courses as they're ready (appetizer, then main) instead of making you wait for everything. This is **detokenization**: converting the model's output token IDs back into a flowing, human-readable string.

The `InputProcessor` prepares the world for the machine, and the `OutputProcessor` prepares the machine's work for the world.

In [ ]:
class SimpleInputProcessor:
    """
    Takes a raw request from a user and prepares it for the engine.
    Its main job is to tokenize the prompt.
    """
    def __init__(self, tokenizer: MockTokenizer):
        self._tokenizer = tokenizer

    def process_request(self, request: GenerationRequest) -> ProcessedRequest:
        """Tokenizes the prompt string into a list of integers."""
        prompt_token_ids = self._tokenizer.encode(request.prompt)
        
        print(f"[InputProcessor]  Request {request.request_id}: Tokenized prompt.")
        
        return ProcessedRequest(
            request_id=request.request_id,
            prompt_token_ids=prompt_token_ids,
        )


class SimpleOutputProcessor:
    """
    Takes generated token IDs from the engine and converts them back to text.
    It also manages the state for streaming outputs.
    """
    def __init__(self, tokenizer: MockTokenizer):
        self._tokenizer = tokenizer
        # We need to keep track of the tokens generated so far for each request
        self._outputs: Dict[str, List[int]] = {}

    def add_new_request(self, request_id: str):
        """Initializes tracking for a new streaming request."""
        self._outputs[request_id] = []
        
    def append_token(self, request_id: str, token_id: int) -> RequestOutput:
        """Appends a single new token and detokenizes the current output for streaming."""
        if request_id not in self._outputs:
            self.add_new_request(request_id)
            
        self._outputs[request_id].append(token_id)
        
        # Check for the "end of text" token to see if we're done
        finished = (token_id == 0)
        
        # Detokenize the *entire* sequence of generated tokens so far
        generated_text = self._tokenizer.decode(self._outputs[request_id])
        
        if finished:
            # Clean up memory once a request is complete
            del self._outputs[request_id]
        
        return RequestOutput(
            request_id=request_id,
            generated_text=generated_text,
            finished=finished
        )

Let's walk through that simple implementation.

**`SimpleInputProcessor`**
-   It holds an instance of our `MockTokenizer`.
-   Its `process_request` method is straightforward: it receives a `GenerationRequest` containing the raw prompt string.
-   It calls `self._tokenizer.encode()` on the prompt. This is the core tokenization step, turning `"Hello, world"` into `[1, 2, 3]`.
-   It returns a `ProcessedRequest` object, which contains the same `request_id` but now has the `prompt_token_ids` ready for the model engine.

**`SimpleOutputProcessor`**
-   This class is slightly more complex because it needs to handle *state*, especially for streaming.
-   It also holds a `tokenizer`.
-   It uses a dictionary, `self._outputs`, to store the list of generated tokens for each active request. This is crucial; without it, the processor wouldn't know the context of a new incoming token.
-   `add_new_request` simply initializes an empty list for a new request that's about to start generating tokens.
-   `append_token` is the heart of streaming. For a given `request_id`, it:
    1.  Adds the new `token_id` to the list for that request.
    2.  Checks if the token is the special `endoftext` token (`0`), which signals that generation is complete.
    3.  Calls `self._tokenizer.decode()` on the **full list** of tokens generated so far. This is why we need to store the state!
    4.  Returns a `RequestOutput` object containing the currently generated text and a `finished` flag.
    5.  If finished, it cleans up by removing the request ID from its state dictionary.

In [ ]:
# === Demonstration: A Request's Lifecycle ===

# 1. Initialization
tokenizer = MockTokenizer()
input_processor = SimpleInputProcessor(tokenizer)
output_processor = SimpleOutputProcessor(tokenizer)

# 2. A user sends a request
user_request = GenerationRequest(request_id="req-001", prompt="Hello, world!")
print(f"User sent request {user_request.request_id}: '{user_request.prompt}'\n")

# 3. Input processing: The string is turned into token IDs
processed_request = input_processor.process_request(user_request)
print(f"--> Input for engine: {processed_request.prompt_token_ids}\n")

# 4. Simulate the engine generating tokens one by one (streaming)
# The engine would produce these token IDs after processing the prompt.
# 0 is the <|endoftext|> token.
generated_token_ids = [10, 11, 12, 13, 14, 15, 16, 17, 18, 9, 0] 
print("[Engine]          Starting generation for req-001...")

# 5. Output processing: The token IDs are turned back into a string, piece by piece
for token_id in generated_token_ids:
    # The engine sends one token at a time to the output processor
    stream_output = output_processor.append_token(
        request_id=processed_request.request_id,
        token_id=token_id,
    )
    
    # We receive the detokenized text at each step
    print(f"[OutputProcessor] Stream for {stream_output.request_id}: '{stream_output.generated_text}' (Finished: {stream_output.finished})")

print("\nGeneration complete!")

### Going Deeper: Why Validate at the Input Stage?

If you look at the real `vllm.engine.input_processor.py` source code, you'll see a tremendous amount of code dedicated to validation (`_validate_params`, `_validate_prompt_len`, etc.). Our simple version skips this entirely, but it's one of the most important functions of a real `InputProcessor`.

**Why not just let the model core handle errors?**

The answer is a core principle of robust systems: **Fail Fast and Fail Cheap.**

-   **GPU time is expensive.** Scheduling a request, allocating KV cache memory (which we'll cover next), and running a model forward pass all consume precious GPU resources.
-   **Input errors are cheap to catch.** Checking if a user's requested `max_tokens` is larger than the model's maximum context length is a simple integer comparison. It costs virtually nothing in terms of computation.

By placing validation at the very beginning of the pipeline in the `InputProcessor`, vLLM ensures that an invalid request is rejected *before* it has a chance to waste any expensive resources. If a user submits a prompt that is already 5000 tokens long and the model only supports 4096, the `InputProcessor` can immediately return an error. The alternative—letting it get scheduled, failing inside the GPU, and then bubbling the error back up—is far more complex and inefficient.

The `InputProcessor` acts as a bouncer at the club door, checking IDs and ensuring rules are followed before letting anyone onto the crowded, expensive dance floor (the GPU).

In [ ]:
# Let's add a simple validation check to our InputProcessor.

class ValidatingInputProcessor:
    """An InputProcessor that also validates requests."""
    def __init__(self, tokenizer: MockTokenizer, max_model_len: int):
        self._tokenizer = tokenizer
        self._max_model_len = max_model_len
        print(f"[InputProcessor]  Initialized with max_model_len = {max_model_len}")

    def process_request(self, request: GenerationRequest) -> ProcessedRequest:
        """Tokenizes and validates the prompt."""
        prompt_token_ids = self._tokenizer.encode(request.prompt)
        
        # --- THE VALIDATION STEP ---
        prompt_len = len(prompt_token_ids)
        if prompt_len >= self._max_model_len:
            # Fail fast! Don't let this request get to the engine.
            raise ValueError(
                f"Prompt length ({prompt_len}) is too long for this model "
                f"(max_model_len: {self._max_model_len})."
            )
        
        print(f"[InputProcessor]  Request {request.request_id}: Validated and tokenized (len={prompt_len}).")
        
        return ProcessedRequest(
            request_id=request.request_id,
            prompt_token_ids=prompt_token_ids,
        )

# --- Demonstration of Validation ---
# Imagine our model has a very small context window of 20 tokens.
validator = ValidatingInputProcessor(tokenizer=MockTokenizer(), max_model_len=20)

# This request is fine
short_request = GenerationRequest(request_id="req-002", prompt="Hello, world!")
try:
    validator.process_request(short_request)
except ValueError as e:
    print(e)

print("-" * 20)

# This one is too long
long_prompt = "The quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog."
long_request = GenerationRequest(request_id="req-003", prompt=long_prompt)
try:
    validator.process_request(long_request)
except ValueError as e:
    print(f"[InputProcessor]  ERROR processing {long_request.request_id}: {e}")

### Visualization: The Tokenization Stream

Text is continuous to us, but discrete to a model. The streaming output process makes this clear. Let's visualize the journey from a string to tokens, and from a stream of tokens back to a string.

In [ ]:
# === Visualization Cell ===

# 1. The prompt comes in as a single string.
prompt = "The quick brown fox."
print(f"Original Prompt: \"{prompt}\"")
print("-" * 30)
print("Stage 1: Input Processing (Tokenization)")
print("-" * 30)

tokenizer = MockTokenizer()
tokens = tokenizer.encode(prompt)
token_map = [f"{token_id:<2d} -> '{tokenizer.inverse_vocab[token_id]}'" for token_id in tokens]

print(f"String         -> Token IDs")
print(f"\"{prompt}\" -> {tokens}")
print("\nToken breakdown:")
for mapping in token_map:
    print(f"- {mapping}")
    time.sleep(0.3)

print("\n" + "=" * 30 + "\n")
print("Stage 2: Output Processing (Streaming Detokenization)")
print("-" * 30)

output_processor = SimpleOutputProcessor(tokenizer)
# Simulate the model generating tokens for the rest of the classic phrase
generated_token_ids = [14, 15, 16, 17, 18, 9, 0]
request_id = "viz-req-1"

for i, token_id in enumerate(generated_token_ids):
    clear_output(wait=True) # This clears the cell output for the animation effect
    print("Stage 2: Output Processing (Streaming Detokenization)")
    print("-" * 30)
    
    new_token_str = tokenizer.decode([token_id])
    output = output_processor.append_token(request_id, token_id)
    
    print(f"Step {i+1}: Engine generated token ID {token_id} ('{new_token_str}')")
    print(f"Cumulative Output: \"{output.generated_text}\"")
    
    time.sleep(0.5)

print("\nFinal Output Complete!")

### Summary

In this notebook, we've demystified the entry and exit points of an LLM serving engine. We saw how raw text from a user is not what the model sees, and how the model's raw output needs careful handling to become useful text again.

**What We Built:**
-   A `SimpleInputProcessor` that mimics tokenization, turning strings into lists of integers.
-   A `SimpleOutputProcessor` that handles detokenization for both simple and streaming responses, correctly managing the state of each ongoing request.
-   A `ValidatingInputProcessor` to demonstrate the crucial "fail fast" principle of production systems.

**Key Takeaways:**
1.  **Translation is Key:** LLMs operate on numbers (tokens), not characters or words directly. The process of tokenization and detokenization is a mandatory translation layer.
2.  **Input is a Gateway:** The `InputProcessor` is more than a tokenizer; it's a gatekeeper that validates requests to protect expensive downstream resources like the GPU.
3.  **Stateful Outputs:** For streaming, the `OutputProcessor` must be stateful, remembering the tokens generated so far for each request to correctly build the output string incrementally.

**What's Next?**
We've prepared our request and we know how the final result will be handled. But what happens in between? The processed request, with its list of token IDs, is passed to the core of the engine. The next major challenge is managing the memory for these requests efficiently on the GPU. This leads us directly to the revolutionary technique at the heart of vLLM: **PagedAttention**.